In [1]:
# pip install opencv-python

In [2]:
import cv2
import numpy as np
import torch

In [3]:
import cv2
import os

def save_keyframes(video_path, output_folder, frame_interval=10, similarity_threshold=0.8):
    # Create output folder if not exists
    os.makedirs(output_folder, exist_ok=True)
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    i = 0
    frame_count = 0
    success, prev_frame = cap.read()
    for _ in range(5):
        if not success:
            print("Video is too short for the specified frame interval.")
            return
        cap.grab()
        frame_count += 1
    while success:
        # Skip frames according to interval
        for _ in range(frame_interval - 1):
            cap.grab()
            frame_count += 1
        
        success, curr_frame = cap.read()
        frame_count += 1
        
        if not success:
            break

        # Convert to grayscale and calculate histograms
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
        
        prev_hist = cv2.calcHist([prev_gray], [0], None, [256], [0, 256])
        curr_hist = cv2.calcHist([curr_gray], [0], None, [256], [0, 256])
        
        # Normalize histograms for better comparison
        cv2.normalize(prev_hist, prev_hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
        cv2.normalize(curr_hist, curr_hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
        
        # Compare histograms
        similarity = cv2.compareHist(prev_hist, curr_hist, cv2.HISTCMP_CORREL)
        
        # Save frame if significant change detected
        if similarity < similarity_threshold:
            i += 1
            output_path = os.path.join(output_folder, f"keyframe_{i}.jpg")
            cv2.imwrite(output_path, prev_frame)
            print(f"Saved keyframe {i} (Frame {frame_count})")
        
        prev_frame = curr_frame

    cap.release()
    print(f"Finished processing. Total keyframes saved: {i}")

In [4]:
# def save_keyframes(video_path, output_folder):
#     videoCapture = cv2.VideoCapture(video_path)
#     success, frame = videoCapture.read()
#     i = 0
#     while success:
#         gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
#         hist = cv2.calcHist([gray_frame], [0], None, [256], [0, 256])
        
#         success, next_frame = videoCapture.read()
#         if not success:
#             break
        
#         next_gray_frame = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)
        
#         next_hist = cv2.calcHist([next_gray_frame], [0], None, [256], [0, 256])
        
#         similarity = cv2.compareHist(hist, next_hist, cv2.HISTCMP_CORREL)
        
#         if similarity < 0.9:
#             i += 1
#             cv2.imwrite(f"{output_folder}/keyframe_{i}.jpg", frame)
#             print(f"Saved keyframe {i}")
        
#         frame = next_frame

#     videoCapture.release()

In [5]:
save_keyframes('../video/test.mp4', '../output')

Saved keyframe 1 (Frame 25)
Saved keyframe 2 (Frame 35)
Saved keyframe 3 (Frame 45)
Saved keyframe 4 (Frame 165)
Saved keyframe 5 (Frame 375)
Saved keyframe 6 (Frame 465)
Saved keyframe 7 (Frame 795)
Saved keyframe 8 (Frame 945)
Saved keyframe 9 (Frame 1005)
Saved keyframe 10 (Frame 1245)
Saved keyframe 11 (Frame 1815)
Saved keyframe 12 (Frame 2055)
Saved keyframe 13 (Frame 2065)
Saved keyframe 14 (Frame 2215)
Saved keyframe 15 (Frame 2255)
Saved keyframe 16 (Frame 2275)
Saved keyframe 17 (Frame 2285)
Saved keyframe 18 (Frame 2475)
Saved keyframe 19 (Frame 2755)
Saved keyframe 20 (Frame 2765)
Saved keyframe 21 (Frame 2775)
Saved keyframe 22 (Frame 2785)
Saved keyframe 23 (Frame 2865)
Saved keyframe 24 (Frame 2905)
Saved keyframe 25 (Frame 2915)
Saved keyframe 26 (Frame 2925)
Saved keyframe 27 (Frame 3115)
Saved keyframe 28 (Frame 3175)
Saved keyframe 29 (Frame 3405)
Saved keyframe 30 (Frame 3545)
Saved keyframe 31 (Frame 3565)
Saved keyframe 32 (Frame 3575)
Saved keyframe 33 (Frame 365

In [6]:
from PIL import Image
import requests, base64
import os
import bitsandbytes
print(bitsandbytes.__version__)  # Should show ≥0.39.0


0.46.0


In [7]:
images = [] 
placeholder = "" 
for i in range(1,5): 
    with open("../output/keyframe_"+str(i)+".jpg", "rb") as f:

        images.append(Image.open("../output/keyframe_"+str(i)+".jpg"))
        placeholder += f"<|image_{i}|>\n"
        # print(i)

In [8]:
images

[<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1280x720>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1280x720>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1280x720>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1280x720>]

In [9]:
from transformers import AutoModelForCausalLM 
from transformers import AutoProcessor

In [10]:
model_id = "microsoft/Phi-3-vision-128k-instruct"

In [11]:
import torch
torch.cuda.empty_cache()

In [12]:
model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Phi-3-vision-128k-instruct",
        device_map="auto",
        offload_folder="offload",  # Thư mục tạm cho các layer offload
        torch_dtype=torch.float16,
        trust_remote_code=True,
        max_memory={
            0: "6GiB",  # Tối đa 6GB VRAM
            "cpu": "12GiB"  # Sử dụng 12GB RAM 
        },
        _attn_implementation="eager"
        
    )

d:\miniconda\envs\ai\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

d:\miniconda\envs\ai\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some parameters are on the meta device because they were offloaded to the cpu.


In [13]:
messages = [
                {"role": "user", "content": placeholder+"Summarize the video."}, 
]

In [14]:
from transformers import AutoModelForCausalLM 
from transformers import AutoProcessor


# from image_embedding_phi3_v import Phi3VImageProcessor 

# transformers.Phi3VImageProcessor = Phi3VImageProcessor 

In [15]:
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True, num_crops=4)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [16]:
prompt = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [17]:
processor.image_processor.size = {"height": 224, "width": 224}  # Giảm độ phân giải
processor.image_processor.do_resize = True

In [18]:
images = [] 
placeholder = "" 
for i in range(1,5): 
    with open("../output/keyframe_"+str(i)+".jpg", "rb") as f:

        images.append(Image.open("../output/keyframe_"+str(i)+".jpg"))
        placeholder += f"<|image_{i}|>\n"
        # print(i)
# Khi sử dụng processor
inputs = processor(
    prompt, 
    images, 
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to("cuda:0")

In [19]:
generation_args = { "max_new_tokens": 1000, "temperature": 0.0, "do_sample": False, }

In [20]:
generate_ids = model.generate(**inputs, eos_token_id=processor.tokenizer.eos_token_id, **generation_args)

d:\miniconda\envs\ai\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\Danh BH\.cache\huggingface\modules\transformers_modules\microsoft\Phi-3-vision-128k-instruct\c45209e90a4c4f7d16b2e9d48503c7f3e83623ed\image_embedding_phi3_v.py:197: UserWarning: Phi-3-V modifies `input_ids` in-place and the tokens indicating images will be removed after model forward. If your workflow requires multiple forward passes on the same `input_ids`, please make a copy of `input_ids` before passing it to the model.
  warnings.warn(
You are not running the flash-attention implementation, expect numerical differences.


In [21]:
generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]

In [22]:
response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

In [23]:
response

'The image shows a close-up of a spider with its body illuminated by a blue light, possibly in a dark environment. The spider appears to be in a resting or predatory state, with its legs spread out and its body facing the camera.'